# Basketball shooting simulation

Two players each take **20 shots**. Player A has probability **63%** of making each shot; Player X has **65%**. We simulate the game **1 million times** and estimate the probability that **Player X** (p = 0.65) wins (makes more baskets).

In [8]:
import numpy as np

np.random.seed(42)

p_A = 0.63   # Player A: 63% chance per shot
p_X = 0.65   # Player X: 65% chance per shot (the one we care about)
n_shots = 20
n_games = 1_000_000

# Each game: number of baskets for each player (binomial)
baskets_A = np.random.binomial(n_shots, p_A, size=n_games)
baskets_X = np.random.binomial(n_shots, p_X, size=n_games)

# Player X wins when they make more baskets than Player A
x_wins = (baskets_X > baskets_A).sum()
ties = (baskets_X == baskets_A).sum()
a_wins = (baskets_A > baskets_X).sum()

prob_x_wins = x_wins / n_games

print(f"Simulated {n_games:,} games (each player takes {n_shots} shots).")
print(f"Player A (p = {p_A:.0%}): wins {a_wins:,}  |  ties {ties:,}  |  Player X (p = {p_X:.0%}): wins {x_wins:,}")
print(f"\nProbability that Player X (p = 0.65) wins: {prob_x_wins:.4f}  ({prob_x_wins:.2%})")

Simulated 1,000,000 games (each player takes 20 shots).
Player A (p = 63%): wins 381,996  |  ties 130,101  |  Player X (p = 65%): wins 487,903

Probability that Player X (p = 0.65) wins: 0.4879  (48.79%)


---
## Possession game: 40 total shots, rebound rules

- **40 total shot attempts** in the game (shared).
- **Player X (p = 0.65)** starts with possession.
- **Make** → same player keeps the ball (shoots again).
- **Miss** → rebound: **Player A (p = 0.63)** has **70%** chance to get the ball, **Player X** has **30%**.

We simulate 1 million games and report the probability that Player X wins (more baskets than A).

In [9]:
np.random.seed(42)

p_A = 0.63
p_X = 0.65
p_rebound_A = 0.60   # on any miss, A gets ball with 70%, X with 30%
n_shots_total = 40
n_games = 1_000_000

score_A = np.zeros(n_games, dtype=int)
score_X = np.zeros(n_games, dtype=int)
possession = np.ones(n_games, dtype=int)   # 1 = X has ball, 0 = A has ball (X starts)

for _ in range(n_shots_total):
    # X's turn (possession == 1)
    x_turn = possession == 1
    r = np.random.random(n_games)
    x_makes = x_turn & (r < p_X)
    score_X[x_makes] += 1
    x_misses = x_turn & ~(r < p_X)
    rebound = np.random.random(n_games)
    possession[x_misses] = np.where(rebound[x_misses] < p_rebound_A, 0, 1)

    # A's turn (possession == 0)
    a_turn = possession == 0
    r2 = np.random.random(n_games)
    a_makes = a_turn & (r2 < p_A)
    score_A[a_makes] += 1
    a_misses = a_turn & ~(r2 < p_A)
    rebound2 = np.random.random(n_games)
    possession[a_misses] = np.where(rebound2[a_misses] < p_rebound_A, 0, 1)

x_wins = (score_X > score_A).sum()
ties = (score_X == score_A).sum()
a_wins = (score_A > score_X).sum()
prob_x_wins = x_wins / n_games

print(f"Possession game: {n_shots_total} total shots, X starts, rebound 70% A / 30% X on miss.")
print(f"Simulated {n_games:,} games.")
print(f"Player A (p = {p_A:.0%}): wins {a_wins:,}  |  ties {ties:,}  |  Player X (p = {p_X:.0%}): wins {x_wins:,}")
print(f"\nProbability that Player X (p = 0.65) wins: {prob_x_wins:.4f}  ({prob_x_wins:.2%})")

Possession game: 40 total shots, X starts, rebound 70% A / 30% X on miss.
Simulated 1,000,000 games.
Player A (p = 63%): wins 581,166  |  ties 33,871  |  Player X (p = 65%): wins 384,963

Probability that Player X (p = 0.65) wins: 0.3850  (38.50%)


---
## Lakers vs Nuggets (100 games)

- **Lakers**: 82 shots. 3pt rate 40.6%, make 35.7%; 2pt rate 59.4%, make 59%.
- **Nuggets**: 85 shots. 3pt rate 40.3%, make 39.2%; 2pt rate 59.7%, make 56.1%.

Each shot is 3pt or 2pt by the team's mix; make with the team's rate; score 3 or 2 points. Simulate 100 games and report win odds.

In [10]:
np.random.seed(42)

n_games = 100

# Lakers: 82 shots, 3pt 40.6% of attempts @ 35.7%, 2pt 59.4% @ 59%
n_shots_lal = 82
p_3pt_lal, p_make_3pt_lal = 0.406, 0.357
p_2pt_lal, p_make_2pt_lal = 0.594, 0.59

# Nuggets: 85 shots, 3pt 40.3% @ 39.2%, 2pt 59.7% @ 56.1%
n_shots_den = 85
p_3pt_den, p_make_3pt_den = 0.403, 0.392
p_2pt_den, p_make_2pt_den = 0.597, 0.561

# Lakers points per game: for each of 82 shots, 3pt or 2pt, then make or not
is_3pt_lal = np.random.random((n_games, n_shots_lal)) < p_3pt_lal
make_lal = np.random.random((n_games, n_shots_lal))
pts_lal = np.where(is_3pt_lal, (make_lal < p_make_3pt_lal) * 3, (make_lal < p_make_2pt_lal) * 2)
lakers_pts = pts_lal.sum(axis=1)

# Nuggets points per game
is_3pt_den = np.random.random((n_games, n_shots_den)) < p_3pt_den
make_den = np.random.random((n_games, n_shots_den))
pts_den = np.where(is_3pt_den, (make_den < p_make_3pt_den) * 3, (make_den < p_make_2pt_den) * 2)
nuggets_pts = pts_den.sum(axis=1)

lakers_wins = (lakers_pts > nuggets_pts).sum()
nuggets_wins = (nuggets_pts > lakers_pts).sum()
ties = (lakers_pts == nuggets_pts).sum()

print("Lakers vs Nuggets — 100 simulated games")
print(f"  Lakers avg pts: {lakers_pts.mean():.1f}   Nuggets avg pts: {nuggets_pts.mean():.1f}")
print(f"  Lakers wins: {lakers_wins}   Nuggets wins: {nuggets_wins}   Ties: {ties}")
print(f"\nOdds — Lakers: {lakers_wins/n_games:.1%}   Nuggets: {nuggets_wins/n_games:.1%}" + (f"   Tie: {ties/n_games:.1%}" if ties else ""))

Lakers vs Nuggets — 100 simulated games
  Lakers avg pts: 94.4   Nuggets avg pts: 97.6
  Lakers wins: 39   Nuggets wins: 58   Ties: 3

Odds — Lakers: 39.0%   Nuggets: 58.0%   Tie: 3.0%
